# MLflow Hyperparameter Tuning

`Hyperparameter Tuning`:

- the process to **select the optimal configuration settings** for a machine learning model before the training process begins in order to **maximize its predictive accuracy**.


## Environment

In [1]:
from src.tracking import tracking_uri
import sys
from pathlib import Path

import mlflow
import torch

# data
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

# mlflow instance
mlflow.set_tracking_uri(tracking_uri())

# print versions and mlflow
print("torch     ", torch.__version__)
print("cuda      ", torch.cuda.is_available())
print("tracking  ", tracking_uri())
print("experiments", [e.name for e in mlflow.search_experiments()])

torch      2.13.0+cu130
cuda       False
tracking   http://mlflow:5000
experiments ['yolo-plate-detection', '/workspace/runs', 'Default']


## Split data

Define same split before sweep.

- sweep results are comparable based on the same split


In [2]:
from src.data_loader import build_split, verify_split, write_data_yaml

# limit for fast smoke run; e.g. 200
# LIMIT = None  # unlimit
LIMIT = 200  # limit 200

# random seed
SEED = 0

# print split
print(build_split(RAW, PROCESSED, val_fraction=0.2, limit=LIMIT, seed=SEED))
print(verify_split(PROCESSED))

names = (RAW / "classes.txt").read_text().split()
data_yaml = write_data_yaml(ROOT / "configs" / "data.yaml", PROCESSED, names)
print(data_yaml.read_text())

{'train': 160, 'val': 40, 'orphan_images': 0, 'orphan_labels': 0}
{'train': 160, 'val': 40}
path: /workspace/data/processed
train: train/images
val: val/images
nc: 1
names: ['car_plate']



## Define sweep (Grid search)

`Grid search`:

- Exhaustively tests every single **combination in a predefined matrix of values**.


In [3]:
import yaml

# experiment name
EXPERIMENT = "yolo-plate-detection-sweep"

# load base param config
base_cfg = yaml.safe_load((ROOT / "configs" / "train.yaml").read_text())
base_cfg["project"] = str(ROOT / base_cfg["project"])

# define grid
GRID = [{"epochs": e, "name": f"tune-ep{e}"} for e in (10, 20, 30)]

print(f"experiment {EXPERIMENT}")
for g in GRID:
    print(" ", g)

experiment yolo-plate-detection-sweep
  {'epochs': 10, 'name': 'tune-ep10'}
  {'epochs': 20, 'name': 'tune-ep20'}
  {'epochs': 30, 'name': 'tune-ep30'}


## Run sweep

- Each config in grid becomes its own mlflow run
- log with mlflow


In [4]:
from src.tracking import run_sweep

# run sweep with helper function
results = run_sweep(
    grid=GRID,
    base_cfg=base_cfg,
    data_yaml=data_yaml,
    processed_dir=PROCESSED,
    raw_dir=RAW,
    experiment=EXPERIMENT,
    run_name=lambda cfg: f"cpu-ep{cfg['epochs']}-{cfg['imgsz']}px",
)

for r in results:
    print(r)


[1/3] cpu-ep10-416px  {'epochs': 10, 'name': 'tune-ep10'}
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.13.0+cu130 CPU (Intel Core 5 120U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/configs/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0,

2026/08/08 18:21:48 INFO mlflow.tracking.fluent: Experiment with name 'yolo-plate-detection-sweep' does not exist. Creating a new experiment.


MLflow: logging run_id(62fe34c7d0284adb83902a508fba6dde) to http://mlflow:5000
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 416 train, 416 val
Using 0 dataloader workers
Logging results to /workspace/runs/tune-ep10
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10         0G      1.105      3.404      1.024          8        416: 100% ━━━━━━━━━━━━ 20/20 3.5s/it 1:111.6sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0s/it 3.1s2.3s
                   all         40         43    0.00317      0.884      0.325      0.177

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10         0G      1.192      1.929      1.068          7        416: 100% ━━━━━━━━━━━━ 20/20 1.7s/it 33.5s1.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95

## List experiment runs

List emperiment runs for comparison.


In [5]:
from src.tracking import compare_runs

table = compare_runs(EXPERIMENT)
table

,mlflow.runName,epochs,imgsz,data.train_images,metrics/mAP50B,metrics/mAP50-95B,metrics/precisionB,metrics/recallB
0,cpu-ep30-416px,30,416,160,0.974962,0.764988,1.000000,0.906955
1,cpu-ep20-416px,20,416,160,0.970052,0.756008,0.956956,0.906977
2,cpu-ep10-416px,10,416,160,0.951314,0.732104,0.944948,0.906977


visualize historical metrics


In [6]:
import matplotlib.pyplot as plt

# get historical run metrics
client = mlflow.tracking.MlflowClient()
experiment = mlflow.get_experiment_by_name(EXPERIMENT)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for _, row in runs.iterrows():
    label = row["tags.mlflow.runName"]
    for ax, metric in zip(axes, ("metrics/mAP50B", "metrics/mAP50-95B")):
        history = client.get_metric_history(row["run_id"], metric)
        if history:
            ax.plot([p.step for p in history], [
                    p.value for p in history], marker="o", ms=3, label=label)

for ax, metric in zip(axes, ("mAP50", "mAP50-95")):
    ax.set_xlabel("epoch")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

<Figure size 1300x450 with 2 Axes>

In [7]:
# Cost against benefit -- more epochs always helps a little, so the question is
# whether the gain justifies the minutes.
cols = ["tags.mlflow.runName", "params.epochs", "metrics.metrics/mAP50-95B", "metrics.elapsed_seconds"]
cost = runs[[c for c in cols if c in runs.columns]].copy()
cost.columns = [c.split(".", 1)[-1] for c in cost.columns]
cost = cost.sort_values("epochs", key=lambda s: s.astype(int))
cost["min_per_0.01_mAP"] = (
    cost["elapsed_seconds"] / 60 / (cost["metrics/mAP50-95B"] * 100)
).round(2)
cost

,mlflow.runName,epochs,metrics/mAP50-95B,elapsed_seconds,min_per_0.01_mAP
2,cpu-ep10-416px,10,0.732104,441.633076,0.10
1,cpu-ep20-416px,20,0.756008,701.681663,0.15
0,cpu-ep30-416px,30,0.764988,1053.732114,0.23


Browse the runs at http://127.0.0.1:5000.